# Orange di Google Colab: Balancing Dataset Bank Marketing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Raditxt/eksplorasi-knime-orange-n8n/blob/main/colab/01_orange_balancing_di_colab.ipynb)

Notebook ini menjalankan ulang workflow Orange [`orange-bank-marketing.ows`](../orange/) di Google Colab **tanpa GUI**.

Orange bukan hanya aplikasi desktop. Orange juga library Python (`orange3`), jadi widget seperti *Data Sampler*, *Tree*, dan *Python Script* bisa ditulis sebagai kode biasa dan dijalankan di Colab.

**Tujuan:**
1. Menguji apakah hasil di Colab sama dengan hasil di GUI Orange (uji kompatibilitas).
2. Menjalankan eksperimen lanjutan: balancing dengan 15 fitur, bukan hanya 7.

| Widget Orange | Padanan di notebook ini |
|---|---|
| CSV File Import | `pandas.read_csv` → `table_from_frame` |
| Select Columns | pilih kolom di pandas, lalu `Domain` |
| Data Sampler (70%, stratified, seed) | `StratifiedShuffleSplit(random_state=42)`, cara yang sama dengan yang dipakai widget-nya |
| Python Script (undersampling / SMOTE-NC) | fungsi `undersampling()` dan `smote_nc()`, kodenya sama persis |
| Tree | `Orange.classification.TreeLearner` dengan parameter yang sama |
| Test and Score (*Test on test data*) | latih di data sampel, uji di data sisa |

## 1. Persiapan Lingkungan

Notebook ini bisa dijalankan dengan dua cara.

**A. Google Colab.** Colab belum menyediakan Orange dan imbalanced-learn, jadi jalankan sel instalasi di bawah dulu (sekitar 1–2 menit).

**B. Lokal (VS Code / Jupyter) dengan Python bawaan Orange.** Python yang ikut terpasang bersama Orange desktop sudah berisi Orange, jadi **sel instalasi bisa dilewati**. Daftarkan dulu Python tersebut sebagai kernel Jupyter (di PowerShell):

```powershell
& "D:\Orange\python.exe" -m pip install imbalanced-learn
& "D:\Orange\python.exe" -m ipykernel install --user --name orange --display-name "Python (Orange)"
```

Setelah itu pilih kernel **Python (Orange)** di VS Code (*Select Kernel → Jupyter Kernel*).

> Python versi terbaru (misalnya 3.14) belum didukung Orange, sehingga `pip install orange3` bisa sangat lama karena harus meng-compile dari kode sumber. Cara B menghindari masalah ini.

In [1]:
import math
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from imblearn.over_sampling import SMOTENC

import Orange
from Orange.data import Table, Domain
from Orange.data.pandas_compat import table_from_frame
from Orange.classification import TreeLearner

warnings.filterwarnings("ignore")
print("Orange", Orange.__version__)

Orange 3.40.0


## 2. Memuat Dataset

Dataset dibaca langsung dari repositori GitHub, jadi tidak perlu upload manual. File `bank-full.csv` memakai pemisah titik koma (`;`).

In [2]:
URL = "https://raw.githubusercontent.com/Raditxt/eksplorasi-knime-orange-n8n/main/datasets/bank-full.csv"
df = pd.read_csv(URL, sep=";")

# Kolom teks dijadikan kategori supaya Orange membacanya sebagai variabel diskrit
for kolom in df.select_dtypes("object"):
    df[kolom] = df[kolom].astype("category")

print(df.shape)
df.head()

(45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [3]:
distribusi = df["y"].value_counts()
pd.DataFrame({"jumlah": distribusi, "proporsi (%)": (distribusi / len(df) * 100).round(2)})

,jumlah,proporsi (%)
y,,
no,39922,88.3
yes,5289,11.7


Kelas `yes` hanya 11,7% dari data. Kalau tidak ditangani, model cenderung menebak `no` terus dan akurasinya tetap terlihat tinggi.

## 3. Fungsi Bantu

Ketiga fungsi di bawah meniru widget Orange:

- `bagi_data()` meniru **Data Sampler** (70% stratified, seed 42). Widget Orange memakai `StratifiedShuffleSplit` dengan seed tetap 42, jadi cara pembagian di sini sama persis dengannya.
- `undersampling()` dan `smote_nc()` memakai kode yang **sama persis** dengan widget Python Script (lihat [`orange/scripts/`](../orange/scripts/)).

In [4]:
def buat_tabel(df, kolom_fitur, target="y"):
    """Ubah DataFrame menjadi Orange Table dengan target y."""
    t = table_from_frame(df[kolom_fitur + [target]])
    domain = Domain([t.domain[k] for k in kolom_fitur], t.domain[target])
    return t.transform(domain)


def bagi_data(tabel, proporsi=0.7, seed=42):
    """Meniru widget Data Sampler: Fixed proportion, stratified, replicable."""
    n = math.ceil(len(tabel) * proporsi)
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=n,
                                      train_size=len(tabel) - n, random_state=seed)
    idx_sisa, idx_sampel = next(splitter.split(tabel.X, tabel.Y))
    return tabel[idx_sampel], tabel[idx_sisa]   # (Data Sample, Remaining Data)


def undersampling(in_data):
    """Kode yang sama dengan Python Script (undersampling acak)."""
    rng = np.random.default_rng(42)
    y = in_data.Y
    yes = in_data.domain.class_var.values.index("yes")
    idx_yes = np.where(y == yes)[0]
    idx_no = np.where(y != yes)[0]
    keep_no = rng.choice(idx_no, size=len(idx_yes), replace=False)
    keep = np.sort(np.concatenate([idx_yes, keep_no]))
    return in_data[keep]


def smote_nc(in_data):
    """Kode yang sama dengan Python Script (1) (SMOTE-NC)."""
    X = in_data.X
    y = in_data.Y.astype(int)
    cat_idx = [i for i, a in enumerate(in_data.domain.attributes) if a.is_discrete]
    X_res, y_res = SMOTENC(categorical_features=cat_idx, random_state=42).fit_resample(X, y)
    return Table.from_numpy(in_data.domain, X_res, y_res.astype(float))

In [5]:
# Parameter sama dengan widget Tree di workflow Orange
tree = TreeLearner(binarize=True, max_depth=100, min_samples_leaf=2,
                   min_samples_split=5, sufficient_majority=0.95)


def evaluasi(data_latih, data_uji, nama):
    """Meniru Test and Score (Test on test data), kelas positif = yes."""
    model = tree(data_latih)
    pred = model(data_uji)
    asli = data_uji.Y
    yes = data_uji.domain.class_var.values.index("yes")
    tp = int(((pred == yes) & (asli == yes)).sum())
    fp = int(((pred == yes) & (asli != yes)).sum())
    fn = int(((pred != yes) & (asli == yes)).sum())
    tn = int(((pred != yes) & (asli != yes)).sum())
    recall = tp / (tp + fn)
    precision = tp / (tp + fp) if tp + fp else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"Skenario": nama, "Data latih": len(data_latih),
            "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "Recall": recall, "Precision": precision, "F1": f1,
            "Accuracy": (pred == asli).mean()}


def jalankan_tiga_skenario(tabel):
    latih, uji = bagi_data(tabel)
    hasil = [
        evaluasi(latih, uji, "A. Tanpa balancing"),
        evaluasi(undersampling(latih), uji, "B. Undersampling acak"),
        evaluasi(smote_nc(latih), uji, "C. SMOTE-NC"),
    ]
    print(f"Data latih: {len(latih)} baris, data uji: {len(uji)} baris")
    return pd.DataFrame(hasil).set_index("Skenario")


def format_persen(tabel):
    """Tampilkan kolom metrik sebagai persen (tanpa DataFrame.style, supaya tidak butuh jinja2)."""
    hasil = tabel.copy()
    for kolom in hasil.columns:
        nama = kolom[-1] if isinstance(kolom, tuple) else kolom
        if nama in ("Recall", "Precision", "F1", "Accuracy"):
            hasil[kolom] = hasil[kolom].map(lambda v: f"{v:.1%}")
    return hasil

## 4. Eksperimen 1: 7 Fitur (sama dengan workflow Orange)

Kolom yang dipakai sama dengan widget *Select Columns (1)* di Orange dan eksperimen KNIME: `age, job, marital, education, balance, housing, loan`.

Balancing **hanya diterapkan ke data latih**, sedangkan data uji tetap pada distribusi aslinya. Kalau balancing dilakukan sebelum data dibagi, sampel sintetis bisa ikut masuk ke data uji dan hasil evaluasi jadi terlalu optimis.

In [6]:
FITUR_7 = ["age", "job", "marital", "education", "balance", "housing", "loan"]

hasil_7 = jalankan_tiga_skenario(buat_tabel(df, FITUR_7))
format_persen(hasil_7)

Data latih: 31648 baris, data uji: 13563 baris


,Data latih,TP,FP,FN,TN,Recall,Precision,F1,Accuracy
Skenario,,,,,,,,,
A. Tanpa balancing,31648,369,772,1218,11204,23.3%,32.3%,27.1%,85.3%
B. Undersampling acak,7404,930,4406,657,7570,58.6%,17.4%,26.9%,62.7%
C. SMOTE-NC,55892,399,1250,1188,10726,25.1%,24.2%,24.7%,82.0%


### Perbandingan dengan GUI Orange

Angka dari GUI diambil dari widget Test and Score pada workflow `orange-bank-marketing.ows`.

In [7]:
gui_orange = pd.DataFrame({
    "Recall":    [0.233, 0.586, 0.251],
    "Precision": [0.323, 0.174, 0.242],
    "F1":        [0.271, 0.269, 0.247],
    "Accuracy":  [0.853, 0.627, 0.820],
}, index=hasil_7.index)

perbandingan = pd.concat(
    {"GUI Orange": gui_orange, "Colab": hasil_7[gui_orange.columns].round(3)}, axis=1
)
format_persen(perbandingan)

GUI Orange                            Colab            \
                          Recall Precision     F1 Accuracy Recall Precision   
Skenario                                                                      
A. Tanpa balancing         23.3%     32.3%  27.1%    85.3%  23.3%     32.3%   
B. Undersampling acak      58.6%     17.4%  26.9%    62.7%  58.6%     17.4%   
C. SMOTE-NC                25.1%     24.2%  24.7%    82.0%  25.1%     24.2%   

                                       
                          F1 Accuracy  
Skenario                               
A. Tanpa balancing     27.1%    85.3%  
B. Undersampling acak  26.9%    62.7%  
C. SMOTE-NC            24.7%    82.0%

**Hasil uji kompatibilitas:**

| Lingkungan | A. Tanpa balancing | B. Undersampling | C. SMOTE-NC |
|---|---|---|---|
| GUI Orange (Windows) | identik | identik | identik (acuan) |
| Notebook, Python bawaan Orange (Windows) | identik | identik | **identik** (F1 24,7%) |
| Notebook, Google Colab (Linux) | identik | identik | selisih ±0,4 poin (F1 24,3%) |

- **Notebook ini mereproduksi GUI Orange 100%** jika dijalankan di lingkungan yang sama (Windows, Python bawaan Orange). Artinya widget Data Sampler, Tree, dan kedua Python Script berhasil diterjemahkan ke kode dengan tepat.
- **Di Colab, SMOTE-NC selisih sekitar 0,4 poin persen**, sedangkan skenario A dan B tetap identik. Penyebabnya bukan versi scikit-learn (sudah diuji dengan versi 1.5.2 yang sama) dan bukan cara membaca data (kodenya sama). Yang berbeda adalah **lingkungannya** (Linux vs Windows). SMOTE-NC mencari tetangga terdekat, dan pada atribut kategorikal banyak data yang jaraknya persis sama, sehingga urutan pemilihan di antara data yang seri bisa berbeda antar platform.
- Selisih ini **tidak mengubah kesimpulan**. Untuk teknik yang melibatkan pencarian tetangga terdekat, hasil bisa sedikit berbeda antar platform walaupun seed-nya sama.


## 5. Eksperimen 2: 15 Fitur (semua kecuali `duration`)

Pada eksperimen 1, SMOTE-NC tidak lebih baik dari tanpa balancing. Dugaannya, ketujuh fitur itu (data demografis dan keuangan) kurang bisa membedakan kelas `yes` dan `no`, jadi sampel sintetis jatuh di daerah yang tumpang tindih.

Untuk menguji dugaan ini, eksperimen diulang dengan **semua fitur kecuali `duration`**. Kolom `duration` tetap dibuang karena lama telepon baru diketahui setelah telepon selesai (data leakage), sesuai rekomendasi dokumentasi dataset UCI.

In [8]:
FITUR_15 = [k for k in df.columns if k not in ("y", "duration")]
print(FITUR_15)

hasil_15 = jalankan_tiga_skenario(buat_tabel(df, FITUR_15))
format_persen(hasil_15)

['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome']
Data latih: 31648 baris, data uji: 13563 baris


,Data latih,TP,FP,FN,TN,Recall,Precision,F1,Accuracy
Skenario,,,,,,,,,
A. Tanpa balancing,31648,438,715,1149,11261,27.6%,38.0%,32.0%,86.3%
B. Undersampling acak,7404,1003,3833,584,8143,63.2%,20.7%,31.2%,67.4%
C. SMOTE-NC,55892,667,1205,920,10771,42.0%,35.6%,38.6%,84.3%


In [9]:
ringkasan = pd.concat({"7 fitur": hasil_7[["Recall", "Precision", "F1"]],
                       "15 fitur": hasil_15[["Recall", "Precision", "F1"]]}, axis=1)
format_persen(ringkasan)

7 fitur                  15 fitur                 
                       Recall Precision     F1   Recall Precision     F1
Skenario                                                                
A. Tanpa balancing      23.3%     32.3%  27.1%    27.6%     38.0%  32.0%
B. Undersampling acak   58.6%     17.4%  26.9%    63.2%     20.7%  31.2%
C. SMOTE-NC             25.1%     24.2%  24.7%    42.0%     35.6%  38.6%

## 6. Kesimpulan

1. **Orange kompatibel dengan Google Colab.** Workflow GUI bisa ditulis ulang sebagai kode, dan hasilnya sama dengan versi desktop. Di Windows dengan Python bawaan Orange hasilnya identik 100%. Di Colab hanya SMOTE-NC yang selisih sekitar 0,4 poin karena perbedaan platform (lihat bagian 4).
2. **Efek balancing bergantung pada kualitas fitur.** Dengan 7 fitur, SMOTE-NC tidak membantu. Dengan 15 fitur, SMOTE-NC justru memberi F1 tertinggi (sekitar 38%, dibanding 32% tanpa balancing).
3. **Undersampling konsisten menaikkan recall** (sekitar 59–63%), tetapi precision turun. Teknik ini cocok kalau prioritasnya tidak melewatkan nasabah potensial.

**Kelebihan Colab dibanding GUI Orange:** tidak perlu memasang apa pun di laptop, imbalanced-learn tinggal di-`pip install`, eksperimen mudah diulang, dan notebook bisa langsung dibagikan lewat link.

**Kekurangannya:** visualisasi interaktif khas Orange (Distributions, Confusion Matrix yang bisa diklik, dll.) tidak tersedia, sehingga perlu ditulis ulang dengan matplotlib atau pandas.